In [1]:
# Install the required libraries
!pip install -U ultralytics torch torchvision numpy pandas opencv-python-headless

import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from ultralytics import YOLO

print("✅ Infrastructure Loaded.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 3.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 2.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 10.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 8.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 30.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 9.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 2.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 102.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 20.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# --- CONFIGURATION ---
# --- CONFIGURATION ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CRIME_CLASSES = [
    'Abuse', 'Arrest', 'Assault', 'Burglary', 'Fighting', 
    'RoadAccidents', 'Robbery', 'Shooting', 'Shoplifting', 
    'Stealing', 'Vandalism'
]

# Exact training stats for 9-feature normalization
TRAINING_MEAN = np.array([[[0.23507757926049788, 0.20753012595648776, 0.1612644882133125, 
                            -8.239577842805819e-05, -0.00010360057434704374, -5.629274679761271e-05, 
                            -7.270215743652193e-05, -3.938033527811605e-05, -4.5842741051076825e-05]]])

TRAINING_STD = np.array([[[0.5725922961312696, 0.6990307059877556, 0.23833245310698944, 
                           0.5026800639677425, 0.37324910262244865, 0.17072062702789198, 
                           0.8403806621886994, 0.6264200005952775, 0.28410470294664947]]])

# --- NEURAL NETWORK BLUEPRINTS ---

class Attention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        # hidden_size * 2 for Bidirectional
        self.attn = nn.Linear(hidden_size * 2, 1)
        
    def forward(self, x):
        weights = F.softmax(torch.tanh(self.attn(x)), dim=1)
        return torch.sum(weights * x, dim=1)

class GatekeeperV7(nn.Module):
    def __init__(self):
        super().__init__()
        # Updated to 3 layers and 128 hidden size to match your .pth file
        self.lstm = nn.LSTM(9, 128, num_layers=3, batch_first=True, bidirectional=True, dropout=0.4)
        self.attention = Attention(128)
        self.max_pool = nn.AdaptiveMaxPool1d(1)
        
        self.fc = nn.Sequential(
            nn.Linear(256 + 256, 128),
            nn.LeakyReLU(0.1),
            nn.BatchNorm1d(128),
            nn.Dropout(0.5),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid() 
        )

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        attn_out = self.attention(lstm_out)
        pool_out = self.max_pool(lstm_out.transpose(1, 2)).squeeze(-1)
        combined = torch.cat([attn_out, pool_out], dim=1)
        return self.fc(combined)

class SpecialistV1(nn.Module):
    def __init__(self, num_classes=11):
        super().__init__()
        self.lstm = nn.LSTM(9, 128, num_layers=3, batch_first=True, bidirectional=True, dropout=0.4)
        self.attention = Attention(128)
        self.max_pool = nn.AdaptiveMaxPool1d(1)
        
        self.fc = nn.Sequential(
            nn.Linear(256 + 256, 128),
            nn.LeakyReLU(0.1),
            nn.BatchNorm1d(128),
            nn.Dropout(0.5),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        attn_out = self.attention(lstm_out)
        pool_out = self.max_pool(lstm_out.transpose(1, 2)).squeeze(-1)
        combined = torch.cat([attn_out, pool_out], dim=1)
        return self.fc(combined)

print("✅ Architecture Blueprints Updated to match V7 Checkpoints.")

✅ Architecture Blueprints Updated to match V7 Checkpoints.


In [3]:
class RookiePipeline:
    def __init__(self, l1_path, l2_path, mean, std):
        self.mean = mean
        self.std = std
        
        print("1. Waking up Layer 0 (YOLOv8)...")
        self.yolo = YOLO('yolov8n.pt') 
        
        print("2. Waking up Layer 1 (Gatekeeper)...")
        self.l1 = GatekeeperV7().to(DEVICE)
        self.l1.load_state_dict(torch.load(l1_path, map_location=DEVICE))
        self.l1.eval()
        
        print("3. Waking up Layer 2 (Specialist)...")
        self.l2 = SpecialistV1(num_classes=len(CRIME_CLASSES)).to(DEVICE)
        self.l2.load_state_dict(torch.load(l2_path, map_location=DEVICE))
        self.l2.eval()
        
        print("✅ Rookie System is Armed and Ready.")

    def extract_telemetry(self, video_path, target_frames=30):
        """Standard L0 feature extraction."""
        cap = cv2.VideoCapture(video_path)
        features = []
        while len(features) < target_frames and cap.isOpened():
            ret, frame = cap.read()
            if not ret: break
            results = self.yolo(frame, verbose=False)[0]
            classes = results.boxes.cls.cpu().numpy()
            p_count = np.sum(classes == 0)
            v_count = np.sum(np.isin(classes, [2, 3, 5, 7]))
            m_conf = np.max(results.boxes.conf.cpu().numpy()) if len(results.boxes) > 0 else 0
            features.append([p_count, v_count, m_conf])
        cap.release()
        while len(features) < target_frames: features.append([0.0, 0.0, 0.0])
        return np.array(features)

    def process_video(self, video_path, frames_needed=30):
        # Phase 1: Telemetry Extraction
        raw_telemetry = self.extract_telemetry(video_path)
        
        # Phase 2: Feature Engineering (Velocity + Accel)
        v = np.diff(raw_telemetry, axis=0, prepend=raw_telemetry[:1, :])
        a = np.diff(v, axis=0, prepend=v[:1, :])
        combined = np.concatenate([raw_telemetry, v, a], axis=1) # (30, 9)
        
        # Phase 3: Scaling
        combined = combined[np.newaxis, :, :] 
        scaled_data = (combined - self.mean) / (self.std + 1e-6)
        input_tensor = torch.Tensor(scaled_data).to(DEVICE)

        # Phase 4: Dual-Inference
        with torch.no_grad():
            # --- LAYER 1 ---
            l1_output = self.l1(input_tensor)
            # Since V7 has Sigmoid, it returns a value between 0 and 1
            anomaly_score = l1_output.item() 
            
            # --- LAYER 2 ---
            l2_output = self.l2(input_tensor)
            probs = F.softmax(l2_output, dim=1)[0]
            l2_conf, pred_idx = torch.max(probs, 0)
            
            # CRITICAL: These keys must match your print statements exactly
            return {
                "L1_Score": anomaly_score,
                "L1_Decision": "🚨 ANOMALY" if anomaly_score >= 0.5 else "🟢 NORMAL",
                "L2_Prediction": CRIME_CLASSES[pred_idx.item()],
                "L2_Confidence": l2_conf.item(),
                "All_Probs": probs.cpu().numpy()
            }

In [6]:
drone = RookiePipeline(
    l1_path="/kaggle/input/datasets/ritam192/drone-weights/gatekeeper_l1_v7_balanced.pth",
    l2_path="/kaggle/input/datasets/ritam192/drone-weights/specialist_l2_v1.pth",
    mean=TRAINING_MEAN,
    std=TRAINING_STD
)

1. Waking up Layer 0 (YOLOv8)...
2. Waking up Layer 1 (Gatekeeper)...
3. Waking up Layer 2 (Specialist)...
✅ Rookie System is Armed and Ready.


In [7]:
# 1. Path to your test video
TEST_VIDEO_PATH = "/kaggle/input/datasets/ritam192/random/videoplayback (1).mp4" 

# 2. Execution
print(f"🛰️ INITIATING SYSTEM SCAN: {TEST_VIDEO_PATH}")
result = drone.process_video(TEST_VIDEO_PATH)

print("\n" + "="*50)
print("             DRONE ANALYTICS REPORT")
print("="*50)

# Layer 1 Visualization
print(f"[LAYER 1: GATEKEEPER]")
print(f" -> Result: {result['L1_Decision']}")
print(f" -> Anomaly Confidence: {result['L1_Score']*100:.2f}%")

print("-" * 50)

# Layer 2 Visualization
print(f"[LAYER 2: SPECIALIST]")
if result['L1_Score'] < 0.5:
    print(" -> Status: STANDBY (Activity below threat threshold)")
    print(f" -> Hidden Classification: {result['L2_Prediction']} ({result['L2_Confidence']*100:.2f}%)")
else:
    print(f" -> Identified Threat: {result['L2_Prediction']}")
    print(f" -> Identification Confidence: {result['L2_Confidence']*100:.2f}%")

print("="*50)

# Extra: Show Top 3 Likely Crimes (for deeper insight)
print("\nTOP 3 THREAT POSSIBILITIES:")
top_indices = np.argsort(result['All_Probs'])[::-1][:3]
for idx in top_indices:
    print(f" - {CRIME_CLASSES[idx]}: {result['All_Probs'][idx]*100:.2f}%")

🛰️ INITIATING SYSTEM SCAN: /kaggle/input/datasets/ritam192/random/videoplayback (1).mp4

             DRONE ANALYTICS REPORT
[LAYER 1: GATEKEEPER]
 -> Result: 🟢 NORMAL
 -> Anomaly Confidence: 47.62%
--------------------------------------------------
[LAYER 2: SPECIALIST]
 -> Status: STANDBY (Activity below threat threshold)
 -> Hidden Classification: Stealing (30.97%)

TOP 3 THREAT POSSIBILITIES:
 - Stealing: 30.97%
 - Burglary: 22.75%
 - Robbery: 11.11%
